# Gold Layer — agg_restaurant_daily_views (7-day trending, 2026-10-31)

**Grain:** 1 restaurant (`restaurant_id`) × 1 snapshot day (`snapshot_date_key`, Vietnam time), last `KEEP_DAYS` (90) days. **Full overwrite** daily (`CREATE OR REPLACE`). Run after `gold/checks`, before `gold/agg_restaurant_ranking` (which sums it into `credited_views_7d` / `credited_views_prev_7d`).

| Column | Definition |
| --- | --- |
| `credited_views_gained` | `SUM(views_gain × mention_weight)` over the videos that mention the restaurant (split credit). **Additive over days** |
| `video_count_active` | videos of the restaurant whose views went up that day |
| `new_video_count` | videos of the restaurant first seen that day (uploaded ≤ `NEW_VIDEO_DAYS` days before) |

**views_gain of a video on a day** (`fact_video_daily_snapshot`):
- normal day: `views_gained` (views today − views on the previous snapshot; after a missed day it covers the gap, so sums over a window stay right);
- first snapshot of a video uploaded ≤ `NEW_VIDEO_DAYS` (7) days before: `view_count` (all its views are recent, otherwise a new viral video would count 0);
- first snapshot of an older video: 0 (its lifetime views are not "this week");
- negative values (YouTube recounts) are clipped to 0.

In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

DAILY = f"{GOLD}.agg_restaurant_daily_views"
KEEP_DAYS = 90
NEW_VIDEO_DAYS = 7

print(f"Gold source: {GOLD}.fact_video_restaurant_mention, fact_video_daily_snapshot, dim_restaurant")
print(f"Gold target: {DAILY}")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {DAILY}
    COMMENT 'Restaurant x snapshot day: credited views gained (split credit), last {KEEP_DAYS} days, full overwrite daily. credited_views_gained is additive over days.'
    AS
    WITH latest_day AS (
        SELECT CAST(date_format(date_sub(to_date(CAST(MAX(snapshot_date_key) AS STRING), 'yyyyMMdd'), {KEEP_DAYS - 1}), 'yyyyMMdd') AS INT) AS from_key
        FROM {GOLD}.fact_video_daily_snapshot
    ),
    video_gain AS (
        SELECT f.snapshot_date_key, f.video_key,
               greatest(CASE
                   WHEN NOT f.is_first_snapshot                       THEN COALESCE(f.views_gained, 0)
                   WHEN COALESCE(f.video_age_days, 9999) <= {NEW_VIDEO_DAYS} THEN COALESCE(f.view_count, 0)
                   ELSE 0
               END, 0) AS views_gain,
               f.is_first_snapshot AND COALESCE(f.video_age_days, 9999) <= {NEW_VIDEO_DAYS} AS is_new_video
        FROM {GOLD}.fact_video_daily_snapshot f CROSS JOIN latest_day l
        WHERE f.snapshot_date_key >= l.from_key AND f.video_key <> -1
    )
    SELECT
        d.restaurant_id,
        g.snapshot_date_key,
        CAST(round(SUM(g.views_gain * m.mention_weight)) AS BIGINT)      AS credited_views_gained,
        CAST(COUNT(DISTINCT CASE WHEN g.views_gain > 0 THEN g.video_key END) AS INT) AS video_count_active,
        CAST(COUNT(DISTINCT CASE WHEN g.is_new_video THEN g.video_key END) AS INT)   AS new_video_count,
        current_timestamp()                                              AS etl_date
    FROM video_gain g
    JOIN {GOLD}.fact_video_restaurant_mention m ON m.video_key = g.video_key AND m.restaurant_key <> -1
    JOIN {GOLD}.dim_restaurant d ON d.restaurant_key = m.restaurant_key
    GROUP BY d.restaurant_id, g.snapshot_date_key
""")

In [ ]:
# --- Verify ---
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id, snapshot_date_key) AS grain,
           COUNT(DISTINCT restaurant_id) AS restaurants, MIN(snapshot_date_key) AS first_day, MAX(snapshot_date_key) AS last_day,
           SUM(credited_views_gained) AS total
    FROM {DAILY}
""").collect()[0]
print(f"📊 {DAILY}: {s['n']} rows, {s['restaurants']} restaurants, {s['first_day']} → {s['last_day']}, "
      f"{s['total']} credited views gained " + ("✅" if s["n"] == s["grain"] else "❌ duplicate grain"))
assert s["n"] == s["grain"], "duplicate restaurant x day rows"
display(spark.sql(f"""
    SELECT snapshot_date_key, COUNT(*) AS restaurants, SUM(credited_views_gained) AS credited_views_gained,
           SUM(new_video_count) AS new_videos
    FROM {DAILY} GROUP BY 1 ORDER BY 1 DESC LIMIT 14
"""))
print("\n--- agg_restaurant_daily_views complete ---")